In [16]:
import numpy as np
import pandas as pd

from numpy.lib.format import open_memmap
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


def stream_clean_chunks(csv_files, chunksize=50_000):
    """Yield cleaned feature chunks without loading the full dataset."""
    for path in csv_files:
        header = pd.read_csv(path, nrows=0)
        label_column = next(
            column for column in header.columns
            if column.strip().lower() == "label"
        )

        for chunk in pd.read_csv(
            path,
            dtype={label_column: "string"},
            chunksize=chunksize,
            low_memory=False,
        ):
            chunk.columns = chunk.columns.str.strip()
            label_column_clean = label_column.strip()
            labels = (
                chunk[label_column_clean]
                .astype("string")
                .str.strip()
                .str.upper()
                .ne("BENIGN")
                .astype(np.int8)
            )

            features = chunk.drop(columns=[label_column_clean])
            features = features.replace([np.inf, -np.inf], np.nan)

            for column in features.columns:
                if not pd.api.types.is_numeric_dtype(features[column]):
                    features[column] = pd.to_numeric(
                        features[column],
                        errors="coerce",
                    )

            features = features.select_dtypes(include=[np.number])
            features = features.astype(np.float32)
            yield features, labels


def split_masks(labels, rng):
    """Create reproducible approximately stratified 70/15/15 masks."""
    values = labels.to_numpy()
    random_values = np.empty(len(values), dtype=np.float32)

    for label in np.unique(values):
        indices = np.flatnonzero(values == label)
        random_values[indices] = rng.random(len(indices))

    train_mask = random_values < 0.70
    validation_mask = (random_values >= 0.70) & (random_values < 0.85)
    test_mask = random_values >= 0.85
    return train_mask, validation_mask, test_mask

In [17]:
from pathlib import Path
from joblib import dump

project_root = Path.cwd()
if not (project_root / "data" / "raw").exists():
    project_root = project_root.parent

raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
csv_files = sorted(raw_dir.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {raw_dir}")

print(f"Found {len(csv_files)} files")

# Fit the imputer on a bounded sample to avoid keeping the full dataset in memory.
sample_parts = []
sample_rows = 0
sample_limit = 100_000

for features_chunk, _ in stream_clean_chunks(csv_files):
    remaining = sample_limit - sample_rows
    if remaining <= 0:
        break
    sample = features_chunk.iloc[:remaining]
    sample_parts.append(sample)
    sample_rows += len(sample)

imputer = SimpleImputer(strategy="median")
imputer.fit(pd.concat(sample_parts, ignore_index=True))
feature_names = sample_parts[0].columns.tolist()
del sample_parts

# Count the reproducible stratified split and fit the scaler incrementally.
seed = 42
split_counts = {"train": 0, "validation": 0, "test": 0}
scaler = StandardScaler()
rng = np.random.default_rng(seed)

for features_chunk, labels_chunk in stream_clean_chunks(csv_files):
    transformed = imputer.transform(features_chunk).astype(np.float32)
    train_mask, validation_mask, test_mask = split_masks(labels_chunk, rng)

    scaler.partial_fit(transformed[train_mask])
    split_counts["train"] += int(train_mask.sum())
    split_counts["validation"] += int(validation_mask.sum())
    split_counts["test"] += int(test_mask.sum())

print(f"Split counts: {split_counts}")

# Allocate .npy files and fill them chunk by chunk.
arrays = {}
for name, count, dtype in [
    ("x_train", split_counts["train"], np.float32),
    ("x_validation", split_counts["validation"], np.float32),
    ("x_test", split_counts["test"], np.float32),
    ("y_train", split_counts["train"], np.int8),
    ("y_validation", split_counts["validation"], np.int8),
    ("y_test", split_counts["test"], np.int8),
]:
    shape = (count, len(feature_names)) if name.startswith("x_") else (count,)
    arrays[name] = open_memmap(
        processed_dir / f"{name}.npy",
        mode="w+",
        dtype=dtype,
        shape=shape,
    )

write_positions = {name: 0 for name in arrays}
rng = np.random.default_rng(seed)

for features_chunk, labels_chunk in stream_clean_chunks(csv_files):
    transformed = imputer.transform(features_chunk).astype(np.float32)
    transformed = scaler.transform(transformed).astype(np.float32)
    train_mask, validation_mask, test_mask = split_masks(labels_chunk, rng)

    for split_name, mask in [
        ("train", train_mask),
        ("validation", validation_mask),
        ("test", test_mask),
    ]:
        start = write_positions[f"x_{split_name}"]
        end = start + int(mask.sum())
        arrays[f"x_{split_name}"][start:end] = transformed[mask]
        arrays[f"y_{split_name}"][start:end] = labels_chunk.to_numpy()[mask]
        write_positions[f"x_{split_name}"] = end
        write_positions[f"y_{split_name}"] = end

for array in arrays.values():
    array.flush()

artifacts = {
    "imputer": imputer,
    "scaler": scaler,
    "feature_names": feature_names,
    "split_counts": split_counts,
}
dump(artifacts, processed_dir / "preprocessor.joblib")

assert all(np.isfinite(array).all() for name, array in arrays.items() if name.startswith("x_"))
assert set(np.unique(arrays["y_train"])).issubset({0, 1})

print(f"Train features: {arrays['x_train'].shape}")
print(f"Validation features: {arrays['x_validation'].shape}")
print(f"Test features: {arrays['x_test'].shape}")
print(f"Saved Phase 2 artifacts to {processed_dir}")

Found 8 files
Split counts: {'train': 1981835, 'validation': 424601, 'test': 424307}
Train features: (1981835, 78)
Validation features: (424601, 78)
Test features: (424307, 78)
Saved Phase 2 artifacts to d:\FedGuard-XAI\data\processed
